# Projeto Titanic — Previsão de Sobrevivência
# Titanic Project — Survival Prediction

Modelo de classificação para prever a sobrevivência dos passageiros do Titanic.

Classification model to predict Titanic passenger survival.

> **Variável-alvo / Target variable:** `Survived` — `1` representa sobrevivência / represents survival, e `0` representa não sobrevivência / represents non-survival.

In [1]:
# importando bibliotecas utilizadas
# importing used libraries
import pandas as pd
from sklearn import tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

In [2]:
# lendo os dados
# reading the data
treino = pd.read_csv("DATA/train.csv")
test = pd.read_csv("DATA/test.csv")

## 1. Inspeção inicial das bases / Initial dataset inspection

`train.csv` contém `Survived`; `test.csv` será usado para as previsões finais.

`train.csv` contains `Survived`; `test.csv` will be used for final predictions.

Visualizando a base de treino e teste.

Visualizing the training and test sets

In [3]:
treino.head(3)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


In [4]:
test.head(3)

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q


## 2. Estrutura e qualidade dos dados / Data structure and quality

`info()` mostra tipos de dados e valores não nulos.

`info()` shows data types and non-null values.

Verificando as informações da base.

Verifying database information

In [5]:
treino.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [6]:
test.info()

<class 'pandas.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-null    int64  
 2   Name         418 non-null    str    
 3   Sex          418 non-null    str    
 4   Age          332 non-null    float64
 5   SibSp        418 non-null    int64  
 6   Parch        418 non-null    int64  
 7   Ticket       418 non-null    str    
 8   Fare         417 non-null    float64
 9   Cabin        91 non-null     str    
 10  Embarked     418 non-null    str    
dtypes: float64(2), int64(4), str(5)
memory usage: 36.1 KB


## 3. Cardinalidade das variáveis / Variable cardinality

Cardinalidade mostra quantos valores distintos há em cada coluna.

Cardinality shows how many distinct values each column has.

Verificando a cardinalidade dos dados.

Checking data cardinality

In [7]:
treino.nunique().sort_values(ascending=False)

PassengerId    891
Name           891
Ticket         681
Fare           248
Cabin          147
Age             88
SibSp            7
Parch            7
Embarked         3
Pclass           3
Survived         2
Sex              2
dtype: int64

In [8]:
test.nunique().sort_values(ascending=False)

PassengerId    418
Name           418
Ticket         363
Fare           169
Age             79
Cabin           76
Parch            8
SibSp            7
Pclass           3
Embarked         3
Sex              2
dtype: int64

## 4. Identificação de valores ausentes / Missing-value identification

Verificamos quais colunas precisam de tratamento de valores ausentes.

We check which columns need missing-value treatment.

Verificando os valores nulos.

checking for null values

In [9]:
treino.isnull().sum().sort_values(ascending=False).head(5)

Cabin          687
Age            177
Embarked         2
PassengerId      0
Name             0
dtype: int64

In [10]:
test.isnull().sum().sort_values(ascending=False).head(5)

Cabin     327
Age        86
Fare        1
Name        0
Pclass      0
dtype: int64

Treino: valores ausentes em `Age` e `Embarked`; teste: em `Age` e `Fare`.

Training: missing values in `Age` and `Embarked`; test: in `Age` and `Fare`.

We have columns containing empty values ​​in the test set that are not empty in the training set (in this case, we will need to handle these columns only in the test set).

## 5. Limpeza e preparação dos dados / Data cleaning and preparation

Removemos colunas pouco úteis e tratamos os valores ausentes.

We remove less useful columns and treat missing values.

Fazendo o tratamento de dados para valores nulos e cardinalidades.

Handling null values ​​and cardinalities during data processing

In [11]:
treino = treino.drop(['Name', 'Ticket', 'Cabin'], axis=1)

In [12]:
test = test.drop(['Name', 'Ticket', 'Cabin'],axis=1)

As colunas `Name`, `Ticket` e `Cabin` são removidas nesta versão do projeto porque possuem muitos valores distintos e exigiriam engenharia de atributos adicional.

The `Name`, `Ticket`, and `Cabin` columns are removed in this version of the project because they have many distinct values and would require additional feature engineering.

Para lidar com valores nulos no campo `Age`, é utilizada a média de idade. Essa é uma imputação simples que preserva a quantidade de registros.

To handle missing values in the `Age` field, the mean age is used. This is a simple imputation that preserves the number of records.

To handle null values ​​in the "age" field, the mean age is being used.

In [13]:
treino.Age.mean()

np.float64(29.69911764705882)

In [14]:
treino.loc[treino.Age.isnull(),'Age'] = treino.Age.mean()

In [15]:
test.loc[test.Age.isnull(),'Age'] = test.Age.mean()

Para `Embarked`, uma variável categórica, utiliza-se a moda: a categoria mais frequente. Para `Fare`, uma variável numérica, utiliza-se a média. Após a imputação, a checagem seguinte confirma que não restaram valores nulos.

For `Embarked`, a categorical variable, the mode is used: the most frequent category. For `Fare`, a numerical variable, the mean is used. After imputation, the next check confirms that no null values remain.

Since there are still two columns with null values—"Embarked" and "Fare"—the mode will be used to fill in the missing values.

In [16]:
treino.Embarked.mode()[0]

'S'

In [17]:
treino.loc[treino.Embarked.isnull(),'Embarked'] = treino.Embarked.mode()[0]

In [18]:
test.loc[test.Fare.isnull(),'Fare'] = test.Fare.mean()

Confirmando se ainda existe valores nulos

Verifying if there are still null values.

In [19]:
treino.isnull().sum().sort_values(ascending=False).head(5)

PassengerId    0
Survived       0
Pclass         0
Sex            0
Age            0
dtype: int64

In [20]:
test.isnull().sum().sort_values(ascending=False).head(5)

PassengerId    0
Pclass         0
Sex            0
Age            0
SibSp          0
dtype: int64

## 6. Codificação das variáveis categóricas / Categorical-variable encoding

`Sex` é convertida em `MaleCheck`: masculino = `1` e feminino = `0`.

`Sex` is converted into `MaleCheck`: male = `1` and female = `0`.

In [21]:
treino.columns[treino.dtypes == 'str']

Index(['Sex', 'Embarked'], dtype='str')

In [22]:
treino.Sex.value_counts()

Sex
male      577
female    314
Name: count, dtype: int64

In [23]:
treino.Embarked.value_counts()

Embarked
S    646
C    168
Q     77
Name: count, dtype: int64

In [24]:
treino['MaleCheck'] = treino.Sex.apply(lambda x: 1 if x == 'male' else 0)

In [25]:
treino.value_counts(['Sex', 'MaleCheck'])

Sex     MaleCheck
male    1            577
female  0            314
Name: count, dtype: int64

In [26]:
test['MaleCheck'] = test.Sex.apply(lambda x: 1 if x == 'male' else 0)

In [27]:
test.value_counts(['Sex', 'MaleCheck'])

Sex     MaleCheck
male    1            266
female  0            152
Name: count, dtype: int64

### Codificação de `Embarked` com One-Hot Encoding / `Embarked` encoding with One-Hot Encoding

`OneHotEncoder` cria uma coluna para cada porto de embarque.

`OneHotEncoder` creates one column for each embarkation port.

Then, the original text columns are removed.

In [28]:
ohe = OneHotEncoder(handle_unknown='ignore', dtype='int32')

In [29]:
ohe = ohe.fit(treino[['Embarked']])

In [30]:
ohe.transform(treino[['Embarked']]).toarray()

array([[0, 0, 1],
       [1, 0, 0],
       [0, 0, 1],
       ...,
       [0, 0, 1],
       [1, 0, 0],
       [0, 1, 0]], shape=(891, 3), dtype=int32)

In [31]:
ohe_df = pd.DataFrame(ohe.transform(treino[['Embarked']]).toarray(),columns=ohe.get_feature_names_out())
ohe_df.head(5)

,Embarked_C,Embarked_Q,Embarked_S
0,0,0,1
1,1,0,0
2,0,0,1
3,0,0,1
4,0,0,1


In [32]:
treino = pd.concat([treino,ohe_df],axis=1)

In [33]:
treino[['Embarked','Embarked_C','Embarked_Q','Embarked_S']].value_counts()

Embarked  Embarked_C  Embarked_Q  Embarked_S
S         0           0           1             646
C         1           0           0             168
Q         0           1           0              77
Name: count, dtype: int64

In [34]:

ohe_df = pd.DataFrame(ohe.transform(test[['Embarked']]).toarray(),columns=ohe.get_feature_names_out())

In [35]:
test = pd.concat([test,ohe_df],axis=1)

In [36]:
test[['Embarked','Embarked_C','Embarked_Q','Embarked_S']].value_counts()

Embarked  Embarked_C  Embarked_Q  Embarked_S
S         0           0           1             270
C         1           0           0             102
Q         0           1           0              46
Name: count, dtype: int64

In [37]:
treino.head(3)

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,MaleCheck,Embarked_C,Embarked_Q,Embarked_S
0,1,0,3,male,22.0,1,0,7.2500,S,1,0,0,1
1,2,1,1,female,38.0,1,0,71.2833,C,0,1,0,0
2,3,1,3,female,26.0,0,0,7.9250,S,0,0,0,1


In [38]:
treino = treino.drop(['Sex','Embarked'],axis=1)

In [39]:
test = test.drop(['Sex','Embarked'],axis=1)

## 7. Definição dos atributos e separação para validação / Feature definition and validation split

`X` contém os atributos; `Y` contém o alvo `Survived`.

`X` contains the features; `Y` contains the `Survived` target.

33% dos dados são reservados para validação; `random_state=42` torna a divisão reproduzível.

33% of the data is reserved for validation; `random_state=42` makes the split reproducible.

In [40]:
X = treino.drop(['PassengerId', 'Survived'],axis= 1)
Y = treino.Survived

In [41]:
X_treino, X_Val, Y_treino, Y_Val = train_test_split(X, Y, test_size=0.33, random_state=42)

## 8. Treinamento e comparação de modelos / Model training and comparison

Três classificadores são treinados com a mesma base: 

- **Árvore de decisão:** aprende regras condicionais a partir dos atributos.
- **KNN:** classifica cada passageiro pela classe predominante entre seus três vizinhos mais próximos.
- **Regressão logística:** estima a probabilidade de sobrevivência a partir das variáveis disponíveis.

Three classifiers are trained with the same dataset:

- **Decision tree:** learns conditional rules from the features.
- **KNN:** classifies each passenger using the predominant class among its three closest neighbors.
- **Logistic regression:** estimates the probability of survival based on the available features.

Após o ajuste, cada modelo prevê `X_Val`.

After fitting, each model predicts `X_Val`.

In [42]:
clf_ac = tree.DecisionTreeClassifier(random_state=42)

In [43]:
clf_ac = clf_ac.fit(X_treino, Y_treino)

In [44]:
y_pred_ac = clf_ac.predict(X_Val)

In [45]:
print(X_treino.dtypes)

Pclass          int64
Age           float64
SibSp           int64
Parch           int64
Fare          float64
MaleCheck       int64
Embarked_C      int32
Embarked_Q      int32
Embarked_S      int32
dtype: object


In [46]:
clf_knn = KNeighborsClassifier(n_neighbors=3)

In [47]:
clf_knn = clf_knn.fit(X_treino,Y_treino)

In [48]:
y_pred_knn = clf_knn.predict(X_Val)

In [49]:
clf_rl = LogisticRegression(random_state=42, max_iter=1000)

In [50]:
clf_rl = clf_rl.fit(X_treino,Y_treino)

In [51]:
y_pred_rl = clf_rl.predict(X_Val)

## 9. Avaliação dos modelos / Model evaluation

A **acurácia** mede os acertos; a **matriz de confusão** detalha os erros.

**Accuracy** measures correct predictions; the **confusion matrix** details the errors.

Essas métricas ajudam a escolher o classificador final.

These metrics help select the final classifier.

In [52]:
accuracy_score(Y_Val, y_pred_ac)

0.7491525423728813

In [53]:
accuracy_score(Y_Val, y_pred_knn)

0.7152542372881356

In [54]:
accuracy_score(Y_Val, y_pred_rl)

0.8169491525423729

In [55]:
confusion_matrix(Y_Val, y_pred_ac)

array([[138,  37],
       [ 37,  83]])

In [56]:
confusion_matrix(Y_Val, y_pred_knn)

array([[147,  28],
       [ 56,  64]])

In [57]:
confusion_matrix(Y_Val, y_pred_rl)

array([[153,  22],
       [ 32,  88]])

In [58]:
X_treino.head(3)

,Pclass,Age,SibSp,Parch,Fare,MaleCheck,Embarked_C,Embarked_Q,Embarked_S
6,1,54.000000,0,0,51.8625,1,0,0,1
718,3,29.699118,0,0,15.5000,1,0,1,0
685,2,25.000000,1,2,41.5792,1,1,0,0


In [59]:
test.head(3)

,PassengerId,Pclass,Age,SibSp,Parch,Fare,MaleCheck,Embarked_C,Embarked_Q,Embarked_S
0,892,3,34.5,0,0,7.8292,1,0,1,0
1,893,3,47.0,1,0,7.0000,0,0,0,1
2,894,2,62.0,0,0,9.6875,1,0,1,0


## 10. Geração das previsões finais / Final prediction generation

A base de teste recebe as previsões de `clf_rl`.

The test set receives predictions from `clf_rl`.

`index=False` não inclui o índice do pandas no CSV.

`index=False` keeps the pandas index out of the CSV.

In [60]:
x_teste = test.drop("PassengerId", axis=1)

In [61]:
y_pred = clf_rl.predict(x_teste)

In [63]:
test['Survived'] = y_pred

In [64]:
base_envio = test[['PassengerId', 'Survived']]

In [65]:
base_envio.to_csv('resultados2.csv', index=False)